# ViSceT5 — Finetune từ checkpoint PRETRAIN
Chạy tuần tự từng cell. Chỉ cần điền **HF token** ở cell cấu hình (mọi thứ khác chạy được luôn).

In [ ]:
!git clone https://github.com/Kussssssss/ViSceT5.git
%cd ViSceT5
!git pull

In [ ]:
!bash setup.sh
# Nếu Colab báo cần restart: Runtime > Restart session, rồi chạy tiếp TỪ cell cấu hình bên dưới
# (KHÔNG cần chạy lại 2 cell clone/setup).

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_xxx'          # <== ĐIỀN token HF của bạn (WRITE để tự upload finetune)
HF_PRETRAIN_REPO = 'Kus669/ViSceT5-pretrain-1epoch'   # repo chứa MODEL ĐÃ PRETRAIN
HF_FINETUNE_REPO = 'Kus669/ViSceT5-finetune'          # repo sẽ lưu model finetune

In [ ]:
import argparse
from scripts import prepare_dataset
prepare_dataset.main(argparse.Namespace(config='configs/data/ViTextVQA.yaml', data_dir='./datasets'))

In [ ]:
from scripts import init_model
init_model.main()

### Tải trọng số pretrain từ HF
Chỉ lấy model cuối ở gốc repo (bỏ optimizer + checkpoint trung gian cho nhẹ).

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id=HF_PRETRAIN_REPO,
    repo_type='model',
    local_dir='/content/pretrain_ckpt',
    allow_patterns=['config.json','generation_config.json','*.safetensors',
                    'spiece.model','tokenizer*.json','special_tokens_map.json','tokenizer_config.json'],
    ignore_patterns=['checkpoint-*/**'],
    token=os.environ['HF_TOKEN'],
)
!ls -lah /content/pretrain_ckpt
assert os.path.exists('/content/pretrain_ckpt/model.safetensors'), \n    'Khong thay model.safetensors o goc repo pretrain — kiem tra lai HF_PRETRAIN_REPO / duong dan checkpoint.'

### Finetune (warm-start từ pretrain)
`--model_name_or_path` nạp trọng số pretrain (strict=False), optimizer/LR khởi tạo mới cho downstream. Chạy in-kernel nên có progress bar.

In [ ]:
import importlib
from training import finetune
importlib.reload(finetune)
finetune.main(args_list=[
    'configs/finetune.yaml',
    '--model_name_or_path', '/content/pretrain_ckpt',
])

### Upload model finetune lên HF

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(repo_id=HF_FINETUNE_REPO, repo_type='model', exist_ok=True)
api.upload_folder(
    folder_path='/content/ViSceT5/output/finetune',
    repo_id=HF_FINETUNE_REPO, repo_type='model',
    ignore_patterns=['checkpoint-*', 'optimizer.pt'],
)
print('Uploaded finetune ->', HF_FINETUNE_REPO)